In [4]:
# 1. Import thư viện
import os
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader, random_split
from PIL import Image
from tqdm import tqdm

print(f"PyTorch version: {torch.__version__}")
print(f"Metal MPS available: {torch.mps.is_available()}")

# Đường dẫn thư mục gốc chứa dữ liệu
# Cấu trúc: /home/user/TACVU2/train, /home/user/TACVU2/public_test, /home/user/TACVU2/private_test
root = "/Users/quangmanh/Project/Olympic-AI-Practice-1/data/TACVU2"


PyTorch version: 2.8.0
Metal MPS available: True


In [6]:
# 2. Thiết lập Seed và Transform

import os
import random
import numpy as np
import torch
import torchvision.transforms as transforms

# Thiết lập Seed để tái lập kết quả
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    print(f"Đã thiết lập seed = {seed}")

SEED = 42
set_seed(SEED)

# Chuẩn hóa ảnh theo chuẩn ImageNet (mean/std)
transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    ),
])


Đã thiết lập seed = 42


In [7]:
# 3. Định nghĩa Dataset

class TrashDataset(Dataset):
    """
    Dataset cho phân loại rác TrashNet 6 classes
    Labels gốc trong CSV: 1-6 (cardboard, glass, metal, paper, plastic, trash)
    PyTorch Labels: 0-5
    """
    def __init__(self, root_dir, csv_file_name=None, transform=None, is_test=False):
        self.root_dir = root_dir
        self.img_dir = os.path.join(root_dir, "images")
        self.transform = transform
        self.is_test = is_test

        if is_test:
            # Test set không có nhãn -> tự quét thư mục ảnh
            self.image_files = sorted([f for f in os.listdir(self.img_dir)
                                      if f.endswith(('.jpg', '.jpeg', '.png'))])
            self.annotations = pd.DataFrame({'file_name': self.image_files})
        else:
            # Train set có file CSV chứa nhãn
            self.annotations = pd.read_csv(os.path.join(root_dir, csv_file_name))

    def __len__(self):
        return len(self.annotations)

    def __getitem__(self, idx):
        row = self.annotations.iloc[idx]
        img_path = os.path.join(self.img_dir, row['file_name'])
        image = Image.open(img_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        if self.is_test:
            return image, -1  # Label giả cho test set

        # Trừ 1 để đưa nhãn 1->6 về chuẩn 0->5 của PyTorch CrossEntropyLoss
        label = int(row['category_id']) - 1
        return image, label

print("Đã định nghĩa TrashDataset")


Đã định nghĩa TrashDataset


In [8]:
# 4. Định nghĩa Model SimpleCNN

import torch
import torch.nn as nn
import torchvision.models as models


class SimpleCNN(nn.Module):
    """
    Mạng CNN đơn giản gồm 2 conv block + Global Average Pooling + Fully Connected
    Input: ảnh RGB 224x224
    Output: logits cho 6 classes
    """
    def __init__(self, num_classes=6, in_ch=3):
        super().__init__()
        # Conv block 1: 3 -> 32 channels, MaxPool giảm spatial xuống 1/2
        self.conv1 = nn.Sequential(
            nn.Conv2d(in_ch, 32, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2)
        )
        # Conv block 2: 32 -> 64 channels
        self.conv2 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True)
        )
        # Global Average Pooling -> vector 64 chiều
        self.gap = nn.AdaptiveAvgPool2d(1)
        # Fully Connected layer cho phân loại
        self.fc = nn.Linear(64, num_classes)

    def forward(self, x):
        x = self.conv1(x)
        x = self.conv2(x)
        x = self.gap(x).flatten(1)
        x = self.fc(x)
        return x


In [11]:
# 5. Tải dữ liệu Train / Validation

# Hyperparameters
BATCH_SIZE = 32
VAL_SPLIT = 0.2  # 20% dữ liệu dùng để validation

# Load toàn bộ tập train từ thư mục /home/user/TACVU2/train
full_dataset = TrashDataset(
    root_dir=os.path.join(root, "train"),
    csv_file_name="train.csv",
    transform=transforms
)

# Chia train/val ngẫu nhiên
val_size = int(len(full_dataset) * VAL_SPLIT)
train_size = len(full_dataset) - val_size
train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])

# Tạo DataLoader
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print(f"Tổng mẫu train: {train_size}, validation: {val_size}")


Tổng mẫu train: 8490, validation: 2122


In [12]:
# 6. Khởi tạo Model và Huấn luyện

# Thiết lập device (GPU nếu có, ngược lại dùng CPU)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Sử dụng device: {device}")

# Khởi tạo model SimpleCNN với 6 classes
model = SimpleCNN(num_classes=6).to(device)

# Optimizer Adam với learning rate 1e-3
optimizer = optim.Adam(model.parameters(), lr=1e-3)

# Hàm loss cho bài toán phân loại nhiều lớp
criterion = nn.CrossEntropyLoss()

# Hyperparameters
NUM_EPOCHS = 10

best_val_loss = float('inf')

best_model_path = os.path.join(os.getcwd(), "best_model.pth")

print(f"Bắt đầu huấn luyện ({NUM_EPOCHS} epochs)...")

# Lưu history để vẽ đồ thị sau
history = {
    'train_loss': [], 'train_acc': [],
    'val_loss': [], 'val_acc': []
}

# Training loop
for epoch in range(NUM_EPOCHS):
    # --- Training phase ---
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS} [Train]"):
        images, labels = images.to(device), labels.to(device)

        # Forward pass
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)

        # Backward pass + cập nhật trọng số
        loss.backward()
        optimizer.step()

        # Thống kê
        running_loss += loss.item() * images.size(0)
        _, predicted = outputs.max(1)
        correct += (predicted == labels).sum().item()
        total += labels.size(0)

    train_loss = running_loss / total
    train_acc = 100 * correct / total

    # --- Validation phase ---
    model.eval()
    val_correct, val_total = 0, 0
    val_running_loss = 0.0

    with torch.no_grad():
        for images, labels in tqdm(val_loader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS} [Val]"):
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)

            val_running_loss += loss.item() * images.size(0)
            _, predicted = outputs.max(1)
            val_correct += (predicted == labels).sum().item()
            val_total += labels.size(0)

    val_loss = val_running_loss / val_total
    val_acc = 100 * val_correct / val_total

    # Lưu history
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)

    # In kết quả mỗi epoch
    print(f"\nEpoch [{epoch+1}/{NUM_EPOCHS}]:")
    print(f"  Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.2f}%")
    print(f"  Val Loss:   {val_loss:.4f}, Val Acc:   {val_acc:.2f}%")

    # Lưu checkpoint khi Val Loss tốt hơn
    if val_loss < best_val_loss:
        print(f"  >> Val Loss cải thiện ({best_val_loss:.4f} -> {val_loss:.4f}). Lưu best model...")
        best_val_loss = val_loss
        torch.save(model.state_dict(), best_model_path)
    print("-" * 50)

print(f"\nHoàn thành huấn luyện! Model tốt nhất lưu tại: {best_model_path}")


Sử dụng device: cpu
Bắt đầu huấn luyện (10 epochs)...


Epoch 1/10 [Val]: 100%|██████████| 67/67 [00:10<00:00,  6.61it/s]



Epoch [1/10]:
  Train Loss: 1.3531, Train Acc: 50.12%
  Val Loss:   1.2630, Val Acc:   57.35%
  >> Val Loss cải thiện (inf -> 1.2630). Lưu best model...
--------------------------------------------------


Epoch 2/10 [Val]: 100%|██████████| 67/67 [00:10<00:00,  6.51it/s]



Epoch [2/10]:
  Train Loss: 1.2019, Train Acc: 55.19%
  Val Loss:   1.1576, Val Acc:   58.25%
  >> Val Loss cải thiện (1.2630 -> 1.1576). Lưu best model...
--------------------------------------------------


...


Epoch [8/10]:
  Train Loss: 1.0620, Train Acc: 60.09%
  Val Loss:   1.0488, Val Acc:   61.88%
--------------------------------------------------


Epoch 9/10 [Val]: 100%|██████████| 67/67 [00:10<00:00,  6.46it/s]



Epoch [9/10]:
  Train Loss: 1.0441, Train Acc: 60.55%
  Val Loss:   1.0626, Val Acc:   61.22%
--------------------------------------------------


Epoch 10/10 [Val]: 100%|██████████| 67/67 [00:10<00:00,  6.53it/s]


Epoch [10/10]:
  Train Loss: 1.0358, Train Acc: 61.30%
  Val Loss:   1.0154, Val Acc:   62.35%
  >> Val Loss cải thiện (1.0373 -> 1.0154). Lưu best model...
--------------------------------------------------

Hoàn thành huấn luyện! Model tốt nhất lưu tại: /Users/quangmanh/Project/Olympic-AI-Practice-1/data/TACVU2/best_model.pth


In [7]:
# 7. Lưu Model và Load Best Checkpoint

import os
import pandas as pd
import torch

# Load lại model tốt nhất (Val Loss thấp nhất) để inference
print("\nChuẩn bị Inference...")
best_model_path = os.path.join(os.getcwd(), "best_model.pth")

# Nạp trọng số tốt nhất vào model
model.load_state_dict(torch.load(best_model_path, map_location=device))

# Chuyển model sang chế độ đánh giá (tắt dropout, batchnorm dùng running stats)
model.eval()

print(f"Đã nạp thành công trọng số tốt nhất từ: {best_model_path}")
print("Mô hình đã sẵn sàng để dự đoán tập Public/Private Test!")



Chuẩn bị Inference...
Đã nạp thành công trọng số tốt nhất từ: /home/user/TACVU2/best_model.pth
Mô hình đã sẵn sàng để dự đoán tập Public/Private Test!


In [8]:
# 8. Inference trên Public Test Set

print("Bắt đầu inference trên Public Test Set...")

# Load Public Test dataset từ /home/user/TACVU2/public_test
public_test_dataset = TrashDataset(
    os.path.join(root, "public_test"),
    transform=transforms,
    is_test=True
)
public_test_loader = DataLoader(
    public_test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

# Inference - không tính gradient để tiết kiệm bộ nhớ
model.eval()
file_names = []
preds_category_id = []

with torch.no_grad():
    for i, (images, _) in enumerate(tqdm(public_test_loader, desc="Public Test")):
        images = images.to(device)
        outputs = model(images)
        _, predicted = outputs.max(1)

        # Lấy file names tương ứng với batch hiện tại
        batch_start = i * BATCH_SIZE
        batch_end = batch_start + images.size(0)
        batch_files = public_test_dataset.annotations.iloc[batch_start:batch_end]['file_name'].values
        file_names.extend(batch_files)

        # Cộng 1 để chuyển nhãn từ 0-5 về 1-6 đúng format nộp bài
        for p in predicted.cpu().numpy():
            preds_category_id.append(int(p) + 1)

# Tạo DataFrame kết quả
df_public = pd.DataFrame({
    'file_name': file_names,
    'category_id': preds_category_id
})

# Lưu file CSV kết quả
public_csv_path = os.path.join(os.getcwd(), "submission.csv")
df_public.to_csv(public_csv_path, index=False)
print(f"Đã tạo file submission.csv cho public test tại {public_csv_path} với {len(df_public)} dự đoán!")

# Hiển thị preview kết quả
print("\nPreview kết quả:")
print(df_public.head(10))


Bắt đầu inference trên Public Test Set...


Public Test: 100%|██████████| 16/16 [00:08<00:00,  1.92it/s]

Đã tạo file submission.csv cho public test tại /home/user/TACVU2/submission.csv với 505 dự đoán!

Preview kết quả:
   file_name  category_id
0  00012.jpg            5
1  00015.jpg            4
2  00017.jpg            3
3  00018.jpg            4
4  00026.jpg            3
5  00028.jpg            5
6  00046.jpg            4
7  00048.jpg            4
8  00054.jpg            4
9  00055.jpg            5


In [9]:
# 9. Inference trên Private Test Set

print("Bắt đầu inference trên Private Test Set...")

# Load Private Test dataset từ /home/user/TACVU2/private_test
private_test_dataset = TrashDataset(
    os.path.join(root, "private_test"),
    transform=transforms,
    is_test=True
)
private_test_loader = DataLoader(
    private_test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

# Inference - không tính gradient để tiết kiệm bộ nhớ
model.eval()
file_names = []
preds_category_id = []

with torch.no_grad():
    for i, (images, _) in enumerate(tqdm(private_test_loader, desc="Private Test")):
        images = images.to(device)
        outputs = model(images)
        _, predicted = outputs.max(1)

        # Lấy file names tương ứng với batch hiện tại
        batch_start = i * BATCH_SIZE
        batch_end = batch_start + images.size(0)
        batch_files = private_test_dataset.annotations.iloc[batch_start:batch_end]['file_name'].values
        file_names.extend(batch_files)

        # Cộng 1 để chuyển nhãn từ 0-5 về 1-6 đúng format nộp bài
        for p in predicted.cpu().numpy():
            preds_category_id.append(int(p) + 1)

# Tạo DataFrame kết quả
df_private = pd.DataFrame({
    'file_name': file_names,
    'category_id': preds_category_id
})

# Lưu file CSV kết quả
private_csv_path = os.path.join(os.getcwd(), "submission.csv")
df_private.to_csv(private_csv_path, index=False)
print(f"Đã tạo file submission.csv cho private test tại {private_csv_path} với {len(df_private)} dự đoán!")

# Hiển thị preview kết quả
print("\nPreview kết quả:")
print(df_private.head(10))


Bắt đầu inference trên Private Test Set...


Private Test: 100%|██████████| 16/16 [00:07<00:00,  2.16it/s]

Đã tạo file submission.csv cho private test tại /home/user/TACVU2/submission.csv với 506 dự đoán!

Preview kết quả:
   file_name  category_id
0  00002.jpg            4
1  00004.jpg            3
2  00006.jpg            1
3  00007.jpg            5
4  00010.jpg            2
5  00014.jpg            4
6  00022.jpg            4
7  00023.jpg            1
8  00024.jpg            4
9  00025.jpg            4
